![DB Academy](https://files.training.databricks.com/binder/prod_binder/ml-online-feature-store-en_us-1.0.0/images/20260910T154812Z/Includes/images/db-academy.png)

### 🚨 PLEASE READ! 🚨
- **When complete, please clean up your resources to avoid hitting workspace limits. Each workspace has a maximum of 10 database instances. Cleaning up prevents resource limits that could block other users in shared Workspaces.**
- To complete this lab you will need to have permission to create a **Lakebase database**.

# Demo - Online Feature Store for Feature Lookups

This demonstration explores the implementation of online feature stores in Databricks, focusing on real-time feature serving and on-demand feature computation. You'll learn to set up online stores, publish feature tables, create Python UDFs for dynamic feature generation, and deploy models that automatically handle feature lookups during inference.

The demo uses a customer churn prediction scenario, demonstrating how to combine pre-materialized features stored in online stores with on-demand features computed at inference time using Python user-defined functions.

## Learning Objectives

By the end of this demonstration, you will be able to:

- Configure and manage online feature stores for low-latency feature access
- Understand prerequisites for publishing feature tables to online stores
- Create Python UDFs for on-demand feature computation
- Publish feature tables to online stores and manage synchronization
- Build training sets that combine online lookups with on-demand features
- Deploy models with automatic feature lookup capabilities
- Inference models registered to Unity Catalog with automatic feature lookups



**🚨 Warning 🚨**: Since `features_online` is protected from deletion at the notebook-level, you will need to leverage the UI to drop the table. If your table `features_online` table already exists inside your `cagtalog.schema`, then you will see the following error when attempting to drop it inside a notebook: `Securable features_online of kind TABLE_ONLINE_VIEW cannot be deleted because it has a parent of kind SCHEMA_STANDARD which is read only.` To troubleshoot this issue and actually drop the table, navigate to the `features_online` in Unity Catalog and use the UI to delete. 
- Search `features_online` in the catalog
- Navigate to the kabob menu in the table menu and click **delete**

**🚨 Warning 🚨**: **If you execute this demo more than once, you will likely run into the above error.** You will need to drop the table from your Lakebase and from Unity Catalog (see above for dropping `features_online` from UC). To drop your table from your Lakebase instance, do the following:
- Navigate to and click on **Compute** in the left menu
- Click on **Lakebase Postgres** and click on the database you want to use if it already exists (e.g., **ml-lakebase-user**).
- Click on **New query** at the top right. This will take you to the SQL editor for postgres queries. 
- Make sure your catalog and schema are selected at the top next to **Run all**. 
- Once you have your catalog and schema selected, drop the table `features_online` using `DROP TABLE features_online`. 

> _You must have both the UC table and the Lakebase table dropped to properly run this demo if you plan on working through it more than once._

## A. Classroom Setup

Run the following cell to configure your working environment for this notebook.

### A1. Compute Requirements

**🚨 REQUIRED - SELECT SERVERLESS COMPUTE**

This course has been configured to run on Serverless compute. While classic compute may also work, testing has been performed on serverless

This demo was tested using version 4 of Serverless compute. To ensure that you are using the correct version of Serverless, please navigate to the **Environment** button on the right and open it (see screenshot below). 

![optional alt text](https://files.training.databricks.com/binder/prod_binder/ml-online-feature-store-en_us-1.0.0/images/20260910T154812Z/Includes/images/serverless-version.png)

### A2. Install Dependencies
Run the next cell to install the necessary libraries and create helper functions for this demonstration. 

In [0]:
%run ./data_set_up

### A3. Setup Lab Environment.

**🚨 NOTE:** You will need to update the following cell to your catalog name. The schema name will be created automatically for you based on the course name.

**🚨 NOTE:** If you are using **Vocareum**, your catalog has already been configured for you and is of the form **labuserXXX_XXX**, which matches your Vocareum username. You should set this as your catalog name.
> Example: catalog_name = "labuser31415926_5358979323"

The catalog and schema variables are used throughout this notebook when referencing Unity Catalog assets.

In [0]:
# Used when needing to pass catalog/schema name with Python
catalog_name = "dbx_catalog"
schema_name = "ml_online_fs"
dev_lab_setup(catalog_name, schema_name) # Store the catalog and schema as catalog_name and schema_name

### A4. Our Data

This demonstration relies on the Telco Customer Dataset from Databricks Marketplace. Note that you may already have access to the Telco Customer Dataset.
#### Vocareum: You have access
If you launched this demo in a Vocareum environemnt, you will automatically have access to the Delta share. It is called `dbacademy_telco_customer_dataset`. For Vocareum users, you will set 
>`databricks_share_name=dbacademy_telco_customer_dataset`. 
#### Non-Vocareum: Check if you have access
Check in the **Catalog Explorer** by searching for `databricks_telco_customer_dataset`. Provided you have the proper level of permisisons on this delta share, you can update the next cell to read 
>`databricks_share_name=databricks_telco_customer_dataset`. 

##### I don't have access/can't see the dataset
If you don't have access or can't see the dataset in your **Catalog Explorer**, the next set of instructions will help walk you through how to get this dataset in your workspace.

1. Navigate to Marketplace and search **Telco Customer Dataset** and click on the tile that reads **Telco Customer Dataset**.
1. Next, click **Get instant access** and follow the on-screen instructions to bring that dataset in.
1. Create a unique Databricks share name. If a name is already in use, you will need to use a different name. Copy the same name into the cell below. For example:
>`databricks_share_name=<unique_name>`. 

In [0]:
## TODO
databricks_share_name = "dbacademy_telco_customer_dataset" # Delta share name

## B. Data Preparation and Feature Engineering Setup

Before working with online feature stores, we need to prepare our datasets and initialize the Feature Engineering client. This section establishes the foundation for feature lookups by loading existing feature and response tables.

The Feature Engineering client provides the primary interface for managing feature tables, online stores, and feature lookups in Databricks.

### B1. Initialize Environment

As a part of the classroom setup, we have created some helper functions to initialize the environment for this demonstration:

- `initialize_uc()`: Sets up Unity Catalog resources for the demo
- `data_preparation()`: Processes the Delta Share assets and prepares feature and response tables

In [0]:
initialize_uc(catalog_name, schema_name)

In [0]:
data_preparation(databricks_share_name, catalog_name, schema_name)

In [0]:
%sql
select * from features limit 10

In [0]:
%sql
select * from response limit 10;

### B2. Initialize Feature Engineering Client

The Feature Engineering client provides the primary interface for managing feature tables, online stores, and feature lookups in Databricks. We'll also load our prepared feature and response tables.

In [0]:
import databricks 
from databricks.feature_engineering import FeatureEngineeringClient

# Initialize the client
fe = FeatureEngineeringClient()

# Define primary key 
primary_key = "customerID"

features_for_online_store = ['TotalCharges', 'MonthlyCharges', 'tenure']

# Read in feature table
feature_table_name = f"{catalog_name}.{schema_name}.features"
X_train_df = fe.read_table(name=feature_table_name)
X_train_pdf = X_train_df.drop(primary_key).toPandas()

# Read in response table 
response_table_name = f"{catalog_name}.{schema_name}.response"
Y_train_df = spark.read.table(response_table_name)
Y_train_pdf = Y_train_df.drop(primary_key).toPandas()

## C. Managing Online Stores

Online feature stores provide low-latency access to feature data for real-time applications and machine learning models. These stores are optimized for high-throughput, low-latency reads required during model inference.


### Instructions
- Update and run the next cell to set your online store name. Note, you can always publish to an existing store if that fits your use case.

The subsequent cell will either get the online store or it will create one, depending on whether it exists or not. 

> The default name for the online store will be `ml-lakebase-<your_custom_name>`.

In [0]:
## TODO
## Fill in the name of your online store
my_online_store = "telco-store"
my_online_store = "ml-lakebase-"+my_online_store

In [0]:
# Get information about an existing online store
store = fe.get_online_store(name=my_online_store)

if store:
    print(f"Store: {store.name}, State: {store.state}, Capacity: {store.capacity}")
else:
    # Create an online store with specified capacity
    fe.create_online_store(
        name=my_online_store,
        capacity="CU_1"  # Valid options: "CU_1", "CU_2", "CU_4", "CU_8"
    )

## D. Prerequisites for Publishing to Online Stores

All feature tables must meet specific requirements before publishing to an online store. These requirements ensure data integrity and enable proper synchronization between offline and online feature stores.

**NOTE:** Failure to meet these prerequisites will prevent successful publication to the online store.

### D1. Required Prerequisites

Before publishing any feature table to an online store, ensure the following requirements are met:

- **Primary key constraint**: Required for online store publishing and data retrieval
- **Non-nullable primary keys**: Primary key columns cannot contain NULL values
- **Change Data Feed enabled**: Required for online store synchronization and incremental updates

In [0]:
%sql 
-- Make sure the primary key columns are not nullable
alter table features alter column customerID set not null;
-- Enable Change Data Feed (CDF)
alter table features set tblproperties (delta.enableChangeDataFeed=true);

### D2. Verifying Prerequisites

Run the next cell to verify that Change Data Feed (CDF) has been enabled on the `features` table. This is essential for maintaining synchronization between the offline feature table and the online store. Also, note that the primary key has been set for `customerID`. 

In [0]:
feature_table_name = f"{catalog_name}.{schema_name}.features"

df = spark.sql(f"select * from {feature_table_name}")

# Ensure Change Data Feed (CDF) is enabled
display(spark.sql(f"describe extended {feature_table_name}"))

## E. Publishing Feature Tables to Online Store

The `publish_table` operation creates a table in the online store, syncs feature data from the offline table, and sets up infrastructure for keeping the online store synchronized with the offline table (note we will be setting the sync type to `triggered` by default).

This process establishes a connection between your offline Delta table and the online PostgreSQL-backed store, enabling low-latency feature retrieval during inference.

🚨 WARNING! 🚨 - depending on how soon you completed the cells above, **_you will likely need to wait ~7-10 minutes for the Lakebase to spin up._** 

### Instructions
- Run the following cell.
- Navigate to **Compute** and click on **Lakebase Postgres** to see the status of your database being created. 
- In the mean time, check out our documentation on Lakebase [here](https://docs.databricks.com/aws/en/oltp/).

> You can run the next cell twice to view the **pipeline_id**. The first time you initialize the publication of the table, you will see `pipeline_id=None`, but the second time it will populate with that value. 

In [0]:
from databricks.ml_features.entities.online_store import DatabricksOnlineStore

# Get the online store instance
online_store = fe.get_online_store(name=my_online_store)

# Publish the feature table to the online store
fe.publish_table(
    online_store=online_store,
    source_table_name=feature_table_name,
    online_table_name=feature_table_name+"_online"
)

### E1. Verifying Online Store Publication

Navigate to **Catalog** and search for `<feature_table_name>_online` (e.g. `features_online`). There you will see your table either in the process of being synced (via a trigger from running the previous cell) or you will see that it has already been synced.

You also find the Lakeflow Spark Declarative Pipeline id (e.g. `412b1b91-daaf-405d-844a-8971a1456885`), where the sync is being handled. Alternatively, you can find the same information in **Jobs & Pipelines** and click on **Pipelines**.

**NOTE:** The data source for this table is **PostgreSQL**, indicating successful publication to the online store.

## F. Creating On-Demand Features with Python UDFs

On-demand features are calculated at inference time using Python user-defined functions (UDFs). These functions are stored in Unity Catalog and can be used during both model training and serving, providing dynamic feature computation capabilities.

This approach is particularly useful for features that depend on real-time data or require complex calculations that are better performed at inference time.

### F1. Creating a Python UDF for Feature Computation

We'll create a Python UDF that calculates the average monthly charges based on total charges and customer tenure. This on-demand feature will be computed dynamically during both training and inference.

**NOTE:** `fe.create_training_set` only supports Python UDFs currently, which will be represented in the SQL command below.

In [0]:
%sql
CREATE OR REPLACE FUNCTION monthly_charges_avrg (TotalCharges DOUBLE, tenure DOUBLE)
RETURNS DOUBLE
LANGUAGE PYTHON AS
$$
avrg = TotalCharges / tenure
return avrg
$$

### F2. Testing the UDF

Let's test our function to ensure it behaves as expected. The result from running the next cell should be **500** (1000 total charges / 2 tenure = 500 average monthly charges).

In [0]:
%sql 
SELECT monthly_charges_avrg(1000, 2) as avg_monthly_charges

### F3. Alternative Approaches: Databricks Function Client

For more advanced use cases, you can use the Databricks Function Client, which provides a specialized interface for creating, managing, and running UC functions in Databricks.

```python
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

# client = DatabricksFunctionClient() # For classic compute
client = DatabricksFunctionClient(execution_mode="serverless") # For serverless compute

sql_query = f"""
CREATE OR REPLACE FUNCTION {catalog_name}.{schema_name}.monthly_charges_avrg (
TotalCharges DOUBLE COMMENT "The total charges",
tenure DOUBLE COMMENT "The tenure of the customer"
)
RETURNS DOUBLE
LANGUAGE PYTHON
COMMENT "Calculates the average monthly charges for a given total charge and tenure."
AS
$$
avrg = TotalCharges / tenure
return avrg
$$
"""
# Create the UDF in Unity Catalog
function_info = client.create_function(sql_function_body=sql_query)

print(f"Created UDF: {function_info.full_name}")
```

## G. Setting Up Feature Lookups and Training Data

Now we'll configure feature lookups that combine pre-materialized features from our online store with on-demand features computed using our Python UDF. This demonstrates the power of hybrid feature serving architectures.

In [0]:
from databricks.feature_engineering import FeatureFunction, FeatureLookup
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

# Define feature lookups including on-demand features
features = [
    # On-demand feature using our Python UDF
    FeatureFunction(
        udf_name=f"{catalog_name}.{schema_name}.monthly_charges_avrg",
        input_bindings={
            "TotalCharges": "TotalCharges",
            "tenure": "tenure"
    },
        output_name="m_charges_avrg",
    ),
    # Lookup pre-materialized features from the online store
    FeatureLookup(
        table_name=feature_table_name,
        feature_names=features_for_online_store,
        lookup_key=primary_key
    )
]

## H. Training a Model with On-Demand Features

In this section, we'll create a training dataset that combines pre-materialized features from our online feature store with on-demand features computed using our Python UDF you just created.

### H1. Creating the Training Set

The training set combines the base DataFrame with features from both online lookups and on-demand computations. The Feature Engineering client automatically handles the feature retrieval and computation process.

In [0]:
# Create training set with feature lookups
training_set = fe.create_training_set(
    df=Y_train_df, 
    feature_lookups=features,
    label='Churn',
    exclude_columns=['customerID']
)

# Load the training data
training_df = training_set.load_df().toPandas()

print("Training dataset columns:", training_df.columns.tolist())
display(training_df.head())

### H2. Training the Model

We'll train a Random Forest classifier using the combined feature set that includes both pre-materialized and on-demand features. This model will automatically handle feature lookups during inference.

In [0]:
# Prepare training data
X_train = training_df.drop(['Churn'], axis=1)
y_train = training_df['Churn']

# Split data for validation
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

# Train the model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_split, y_train_split)

# Evaluate the model
y_pred = model.predict(X_val_split)
accuracy = accuracy_score(y_val_split, y_pred)

print(f"Model Accuracy: {accuracy:.3f}")
print("\nClassification Report:")
print(classification_report(y_val_split, y_pred))

## I. Unity Catalog Model Registration and Deployment

To ensure that our model can automatically evaluate on-demand features during inference, we must log the model using the Feature Store client and register it to Unity Catalog. This preserves the feature lookup metadata and enables automatic feature computation.

### I1. Logging and Registering the Model

Let's examine each component of the model logging process:

- **`mlflow.set_registry_uri("databricks-uc")`**: Ensures that the model automatically evaluates on-demand features when used for inference by setting the registry URI to Unity Catalog
- **`fe.log_model()`**: Logs the model with feature metadata, enabling automatic feature lookup at inference and UC lineage
- `model`: The trained model instance
- `artifact_path`: Location where artifacts are stored under the MLflow Run
- `flavor`: Specifies the MLflow flavor (sklearn in this case)
- `training_set`: The training set created with feature lookups
- `input_example`: Provides a concrete input instance for MLflow signature inference
- `registered_model_name`: Registers the model in UC Model Registry
- `extra_pip_requirements`: Adds Python package requirements for the serving environment

In [0]:
import mlflow

# Set the registry URI to Unity Catalog
mlflow.set_registry_uri("databricks-uc")

# Define model name in Unity Catalog
model_name = f"{catalog_name}.{schema_name}.churn_prediction_model"

# Log the model with feature metadata
fe.log_model(
    model=model,
    artifact_path="model",
    flavor=mlflow.sklearn,
    training_set=training_set,
    input_example=X_train_split.iloc[:1],
    registered_model_name=model_name,
    extra_pip_requirements=["scikit-learn", "numpy", "pandas"]
)

print(f"Model registered as: {model_name}")

### I2. Testing Batch Inference

Let's test our registered model with batch inference to ensure on-demand features are computed correctly. This demonstrates that the model automatically handles feature lookups and UDF computation during scoring.

In [0]:
# Load the registered model
import mlflow.pyfunc

model_version = "1" # Default value when running this notebook the first time

model_uri = f"models:/{model_name}/{model_version}"
loaded_model = mlflow.pyfunc.load_model(model_uri)

# Create a test dataset for batch scoring
test_data_df = spark.createDataFrame(
    [("5575-GNVDE", 957.1, 54.4, 18.0)],
    ["customerID", "TotalCharges", "MonthlyCharges", "tenure"]
)

Note that in our table below, we are only using our features and we have not stored the results from our UC function. This additional computation will be handled in the next section when we make an inference call to our model stored in UC.

In [0]:
display(test_data_df)

### I3. Scoring with Feature Lookups

The `fe.score_batch` function automatically handles feature lookups from the online store and computes on-demand features using our registered UDF. This provides a seamless inference experience that combines multiple feature sources.

In [0]:
from pyspark.sql.types import StringType

predictions = fe.score_batch(
    model_uri=model_uri,
    df=test_data_df,
    result_type=StringType()
)

display(predictions)

## ⚠️ WARNING: Cleanup Instructions (if you don't have time to complete the other related notebooks)

**IMPORTANT:** Please clean up your resources to avoid hitting workspace limits.  
- Each workspace supports a maximum of **10 database instances**.  
- Cleaning up ensures resources are freed and prevents blocking other users in shared environments.

#### Delete Your Lakebase Database Instance
1. Navigate to **Compute** → **Lakebase Postgres**.  
2. Select your **database instance**.  
3. Click **Delete** and confirm.

## Conclusion

This demonstration has shown you how to implement a complete online feature store solution in Databricks. You've learned to:

1. **Configure online feature stores** for low-latency feature access in production environments
2. **Establish prerequisites** including primary key constraints and Change Data Feed for reliable synchronization
3. **Create Python UDFs** for dynamic, on-demand feature computation stored in Unity Catalog
4. **Publish feature tables** to online stores with automatic synchronization capabilities
5. **Build hybrid training sets** that combine pre-materialized and on-demand features
6. **Deploy models** with automatic feature lookup capabilities for seamless inference

The combination of online feature stores and on-demand feature computation provides a powerful, flexible architecture for real-time machine learning applications. This approach enables you to serve features with sub-millisecond latency while maintaining the ability to compute complex features dynamically at inference time.

### Next Steps
Consider exploring model serving endpoints, real-time inference patterns, and advanced feature engineering workflows to further enhance your MLOps capabilities.

### Resources
For further reading, see the [Databricks documentation on online feature stores](https://docs.databricks.com/aws/en/machine-learning/feature-store/online-feature-store#continuously-update-online-features).